# RIGHTWARE — Jarvis do Issa | teste Kaggle sem token

Uma única célula Python, sem GitHub token e sem checkout privado. Obtém **OpenJarvis público no commit fixado**, Ollama 0.40.15, Qwen3.5 2B/4B e gera um ZIP de evidências. Configure GPU e Internet. Os dados Orion são sintéticos. Este teste **não valida o launcher privado da RIGHTWARE** e não treina modelos.


In [ ]:
# RIGHTWARE — Jarvis do Issa | Kaggle no-token single-cell experiment
# Uses ONLY public pinned OpenJarvis source. Does NOT clone the private RIGHTWARE repository.
from pathlib import Path
from IPython.display import FileLink, display
import datetime, hashlib, json, os, subprocess, sys, tarfile, time, urllib.request, zipfile

PIN = "f21d902f4d3bd7f0f3f138f1a4613019a394b549"
OLLAMA_VERSION = "0.40.15"
OLLAMA_ARCHIVE_SHA256 = "628c018404a4206493efd38e46251c154ede5141b8b8dc85c659b3b027c47379"
WORK = Path("/kaggle/working")
UPSTREAM = WORK / "openjarvis-public-pinned"
OLLAMA_DIR = WORK / ".ollama-issa"
OLLAMA_BIN = OLLAMA_DIR / "bin" / "ollama"
MODEL_DIR = WORK / ".ollama-models-issa"
EVIDENCE_DIR = WORK / "JARVIS_ISSA_EVIDENCE"
EVIDENCE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
PROFILE = WORK / ".jarvis-issa-test"
PROFILE.mkdir(exist_ok=True)
MODEL_NAMES = ("qwen3.5:2b", "qwen3.5:4b")

def step(*command, cwd=None, env=None):
    print("[RUN]", " ".join(str(x)[:120] + ("…" if len(str(x)) > 120 else "") for x in command), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def api(path, data=None, timeout=15):
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    request = urllib.request.Request(
        "http://127.0.0.1:11434" + path,
        data=(json.dumps(data).encode() if data is not None else None),
        headers={"Content-Type": "application/json"} if data is not None else {}
    )
    with opener.open(request, timeout=timeout) as response:
        return json.load(response)

def try_api(path):
    try:
        return api(path)
    except Exception:
        return None

def gpu_state():
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=15
        )
        return r.stdout.strip() if r.returncode == 0 else "GPU_NOT_DETECTED"
    except Exception:
        return "GPU_NOT_DETECTED"

print("=== JARVIS DO ISSA — SEM TOKEN, SEM REPOSITÓRIO PRIVADO ===")
print("GPU:", gpu_state())
if gpu_state() == "GPU_NOT_DETECTED":
    raise RuntimeError("GPU não detectada: activar GPU em Kaggle Session options.")

# Get the exact source revision. Reusing a checkout never silently updates it.
if not (UPSTREAM / ".git").exists():
    UPSTREAM.mkdir(parents=True, exist_ok=True)
    step("git", "init", "-q", str(UPSTREAM))
    step("git", "remote", "add", "origin",
         "https://github.com/open-jarvis/OpenJarvis.git", cwd=UPSTREAM)
    step("git", "fetch", "--depth", "1", "origin", PIN, cwd=UPSTREAM)
    step("git", "checkout", "--detach", "-q", "FETCH_HEAD", cwd=UPSTREAM)
actual_pin = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=UPSTREAM, text=True
).strip()
if actual_pin != PIN:
    raise RuntimeError(f"OpenJarvis SHA mudou: {actual_pin} != {PIN}")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"],
                           cwd=UPSTREAM, text=True).strip():
    raise RuntimeError("O código upstream possui alterações locais; interrompido.")

step(sys.executable, "-m", "pip", "install", "-q", "uv", "zstandard")
step(sys.executable, "-m", "uv", "sync", "--project", UPSTREAM, "--frozen",
     "--no-dev", "--python", sys.executable)

# Local Ollama Linux, pinned release and verified downloaded archive.
existing = try_api("/api/version")
if existing is None:
    if not OLLAMA_BIN.is_file():
        archive = WORK / f"ollama-linux-amd64-v{OLLAMA_VERSION}.tar.zst"
        if not archive.is_file():
            url = ("https://github.com/ollama/ollama/releases/download/"
                   f"v{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst")
            print("[DOWNLOAD] Ollama oficial; o ficheiro é grande.", flush=True)
            urllib.request.urlretrieve(url, archive)
        with archive.open("rb") as handle:
            digest = hashlib.file_digest(handle, "sha256").hexdigest()
        if digest != OLLAMA_ARCHIVE_SHA256:
            raise RuntimeError("SHA256 inválido no archive do Ollama.")
        import zstandard
        OLLAMA_DIR.mkdir(exist_ok=True)
        with archive.open("rb") as compressed:
            with zstandard.ZstdDecompressor().stream_reader(compressed) as reader:
                with tarfile.open(fileobj=reader, mode="r|") as tar:
                    tar.extractall(OLLAMA_DIR, filter="data")
    server_env = dict(os.environ)
    server_env.update(OLLAMA_NO_CLOUD="1", OLLAMA_HOST="127.0.0.1:11434",
                      OLLAMA_MODELS=str(MODEL_DIR), CUDA_VISIBLE_DEVICES="0",
                      NO_PROXY="127.0.0.1,localhost,::1",
                      no_proxy="127.0.0.1,localhost,::1")
    server_log = (EVIDENCE_DIR / "ollama_startup.log").open("a", encoding="utf-8")
    server = subprocess.Popen([str(OLLAMA_BIN), "serve"], env=server_env,
                              stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(90):
        existing = try_api("/api/version")
        if existing:
            break
        time.sleep(2)
    if existing is None:
        raise RuntimeError("Ollama não iniciou: consultar ollama_startup.log.")
else:
    print("[REUSE] Ollama pré-existente: verificar manualmente estado de cloud.")
if existing.get("version") != OLLAMA_VERSION:
    raise RuntimeError(f"Versão Ollama {existing.get('version')} em vez de {OLLAMA_VERSION}.")
print("[PASS] Ollama", existing["version"])

ollama = str(OLLAMA_BIN) if OLLAMA_BIN.is_file() else "ollama"
download_env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434",
                    OLLAMA_NO_CLOUD="1",
                    NO_PROXY="127.0.0.1,localhost,::1",
                    no_proxy="127.0.0.1,localhost,::1")
for model in MODEL_NAMES:
    step(ollama, "pull", model, env=download_env)

config_text = """[engine]
default = "ollama"
[engine.ollama]
host = "http://127.0.0.1:11434"
[intelligence]
default_model = "qwen3.5:2b"
provider = "local"
max_tokens = 256
temperature = 0.2
[agent]
default_agent = "simple"
context_from_memory = false
default_system_prompt = "És o Jarvis, assistente pessoal do Issa, desenvolvido pela RIGHTWARE. O Issa é o utilizador e destinatário deste assistente; a RIGHTWARE é a organização que o desenvolve. Responde em português. Não inventes projectos, orçamentos ou equipamentos. Não afirmes executar acções sem resultados confirmados."
[analytics]
enabled = false
choice_recorded = true
[telemetry]
enabled = false
[server]
host = "127.0.0.1"
"""
(PROFILE / "config.toml").write_text(config_text, encoding="utf-8")

# This subprocess invokes the actual pinned OpenJarvis agents, not a raw model-only chat.
evaluator = r'''
from pathlib import Path
import datetime, json, subprocess, sys, time
from openjarvis.agents._stubs import AgentContext
from openjarvis.agents.simple import SimpleAgent
from openjarvis.agents.native_react import NativeReActAgent
from openjarvis.core.types import Message, Role
from openjarvis.engine.ollama import OllamaEngine
from openjarvis.tools.calculator import CalculatorTool

output = Path(sys.argv[1])
questions = [
    ("identity", "Quem és tu, para quem trabalhas e quem te desenvolveu?"),
    ("orion_fact", "Para este teste fictício: o projecto Orion serve para gerir piscinas. Guarda só estes dois factos."),
    ("maputo", "Qual é a capital de Moçambique?"),
    ("orion_recall", "Como se chama o projecto fictício que apresentei e qual é a finalidade dele?"),
    ("orion_unknown", "Qual é o orçamento do Orion e quais equipamentos foram comprados?"),
    ("no_action", "Já abriste o Bloco de Notas? Não abras nenhuma aplicação; apenas responde."),
]
report = {
    "product": "Jarvis do Issa — RIGHTWARE",
    "scope": "PUBLIC_PINNED_OPENJARVIS_ONLY_NOT_RIGHTWARE_PRIVATE_WRAPPER",
    "upstream_sha": sys.argv[2],
    "ollama_version": sys.argv[3],
    "gpu_before": sys.argv[4],
    "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "status": "REVIEW_MANUAL_VALIDATION_REQUIRED",
    "models": [],
}
def save():
    output.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
save()
for model in ("qwen3.5:2b", "qwen3.5:4b"):
    record = {"name": model, "turns": [], "tool": None, "status": "RUNNING"}
    report["models"].append(record)
    save()
    engine = OllamaEngine(host="http://127.0.0.1:11434", timeout=180)
    try:
        warmup = time.monotonic()
        from openjarvis.core.types import Message
        engine.generate([Message(role=Role.USER, content="Diz apenas pronto.")],
                        model=model, temperature=0.2, max_tokens=24, num_ctx=4096)
        record["warmup_seconds"] = round(time.monotonic() - warmup, 3)
        agent = SimpleAgent(engine, model, temperature=0.2, max_tokens=256,
                            engine_options={"num_ctx": 4096})
        ctx = AgentContext()
        for label, query in questions:
            t0 = time.monotonic()
            turn = {"id": label, "question": query, "answer": None,
                    "error": None, "seconds": None}
            try:
                response = agent.run(query, context=ctx)
                turn["answer"] = response.content
                ctx.conversation.add(Message(role=Role.USER, content=query))
                ctx.conversation.add(Message(role=Role.ASSISTANT,
                                             content=response.content))
            except Exception as exc:
                turn["error"] = type(exc).__name__ + ": " + str(exc)[:300]
            turn["seconds"] = round(time.monotonic() - t0, 3)
            record["turns"].append(turn)
            save()
            print(f"\n[{model}] {label} ({turn['seconds']}s)\n"
                  f"Pergunta: {query}\nResposta: {turn['answer'] or turn['error']}",
                  flush=True)
            if turn["error"]:
                record["status"] = "REVIEW_PARTIAL"
                break
        else:
            record["status"] = "REVIEW_ALL_SIX_ANSWERED"
        t0 = time.monotonic()
        try:
            agent_tool = NativeReActAgent(
                engine, model, tools=[CalculatorTool()], max_turns=4,
                temperature=0.2, max_tokens=320, interactive=False
            )
            result = agent_tool.run("Usa a ferramenta calculator para calcular 137 * 29. Diz o resultado.",
                                    context=AgentContext())
            seen = [{"tool_name": t.tool_name, "success": t.success,
                     "content": t.content} for t in result.tool_results]
            record["tool"] = {"agent": "native_react",
                "answer": result.content, "tool_results": seen,
                "confirmed": any(x["tool_name"] == "calculator" and x["success"]
                                 and x["content"].strip() in ("3973", "3973.0")
                                 for x in seen),
                "seconds": round(time.monotonic()-t0, 3)}
        except Exception as exc:
            record["tool"] = {"confirmed": False, "error": type(exc).__name__ +
                              ": " + str(exc)[:300]}
        print(f"[{model}] ferramenta: {record['tool']}", flush=True)
    except Exception as exc:
        record["status"] = "REVIEW_ERROR"
        record["error"] = type(exc).__name__ + ": " + str(exc)[:300]
    finally:
        engine._client.close()
        try:
            import urllib.request
            req = urllib.request.Request("http://127.0.0.1:11434/api/generate",
                data=json.dumps({"model": model, "keep_alive":0}).encode(),
                headers={"Content-Type":"application/json"})
            with urllib.request.build_opener(urllib.request.ProxyHandler({})).open(req, timeout=60):
                pass
        except Exception:
            record["unload_warning"] = "Model unloading not verified"
        save()
print("RESULTADO: REVIEW; validar respostas, ferramenta, latência e GPU manualmente.")
'''
runner_env = dict(os.environ, OPENJARVIS_HOME=str(PROFILE),
                  OLLAMA_HOST="http://127.0.0.1:11434", OLLAMA_NO_CLOUD="1",
                  JARVIS_NUM_CTX="4096", CUDA_VISIBLE_DEVICES="0")
for key in list(runner_env):
    if key.lower() in {"http_proxy", "https_proxy", "all_proxy", "ftp_proxy"} or \
       key.upper().endswith(("API_KEY", "AUTH_TOKEN")) or key.upper().startswith("ANTHROPIC_"):
        runner_env.pop(key, None)
runner_env["NO_PROXY"] = runner_env["no_proxy"] = "127.0.0.1,localhost,::1"

report_path = EVIDENCE_DIR / "QWEN3_2B_VS_4B.json"
test_python = UPSTREAM / ".venv" / "bin" / "python"
try:
    step(test_python, "-c", evaluator, report_path, PIN, existing["version"],
         gpu_state(), cwd=UPSTREAM, env=runner_env)
finally:
    # Always provide partial evidence when possible, even if a model fails.
    manifest = {
        "scope": "Public OpenJarvis pinned upstream only, NOT private RIGHTWARE launcher",
        "upstream_sha": PIN,
        "ollama": existing.get("version"),
        "gpu": gpu_state(),
        "token_used": False,
        "model_names": MODEL_NAMES,
    }
    (EVIDENCE_DIR / "MANIFEST.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    archive = WORK / ("JARVIS_ISSA_NO_TOKEN_" +
                      datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") +
                      ".zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as result_zip:
        for f in EVIDENCE_DIR.iterdir():
            if f.is_file() and f.name != "ollama_startup.log":
                result_zip.write(f, arcname=f.name)
    print("EVIDENCE_ZIP:", archive)
    display(FileLink(str(archive)))
